# IFC Flare and Autoland Diagnostics

Updated for current IFC logging and flare command-chain debugging.

Focus areas:
- sink arrest / balloon timing vs height
- gamma command, theta command, AA director pitch, aircraft pitch relationship
- authority and clamp channels (throttle floor, theta clamp, auth reason/latch)
- TECS energy channels and vertical acceleration margin


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (16, 10)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.max_columns", 240)


In [ ]:
# ===== User config =====
from pathlib import Path

# Auto-detect from repository unless explicitly set.
LOGS_DIR = None
USE_LATEST_LOG = True
LOG_FILE_PATH = Path(r"Integrated Flight Computer/logs/ifc_log_00000135.csv")

VS_NEAR_ZERO_THRESH = 0.10
EARLY_ARREST_GEAR_H = 5.0
BALLOON_VS_THRESH = 0.30
BALLOON_GEAR_H_MIN = 3.0
PERSISTENT_TRACK_ERR_DEG = 2.0
CMD_ACT_PITCH_GAP_WARN = 4.0
AA_CMD_THETA_GAP_WARN = 0.75
NEG_ACCEL_MARGIN_WARN_FRAC = 0.40

ACCEL_SMOOTH_WINDOW = 3
SHOW_MODE_LABELS = True


In [ ]:
def detect_project_root(start: Path) -> Path:
    start = start.resolve()
    for cand in [start] + list(start.parents):
        if (cand / "Integrated Flight Computer" / "logs").exists():
            return cand
    raise FileNotFoundError("Could not find 'Integrated Flight Computer/logs' from current working directory.")


def find_latest_log(logs_dir: Path) -> Path:
    files = sorted(logs_dir.glob("ifc_log_*.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError(f"No IFC log files found in: {logs_dir}")
    return files[0]


def load_ifc_log(csv_path: Path) -> pd.DataFrame:
    raw = pd.read_csv(csv_path, comment="#")
    raw.columns = [c.strip() for c in raw.columns]

    raw["t_s"] = pd.to_numeric(raw.get("t_s", np.nan), errors="coerce")
    df = raw.dropna(subset=["t_s"]).copy()

    categorical_cols = {
        "phase", "subphase", "flare_mode", "flare_auth_reason",
        "asc_validity", "status", "craft_name", "cfg_file"
    }
    for col in df.columns:
        if col in categorical_cols:
            df[col] = df[col].fillna("").astype(str)
        else:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df.sort_values("t_s").reset_index(drop=True)


def safe_col(df: pd.DataFrame, col: str) -> bool:
    return col in df.columns


def col_or_nan(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index)


def phase_segments(df: pd.DataFrame, col: str = "phase"):
    segs = []
    if col not in df.columns or df.empty:
        return segs

    start_idx = 0
    prev = df.iloc[0][col]
    for i in range(1, len(df)):
        cur = df.iloc[i][col]
        if cur != prev:
            segs.append((str(prev), df.iloc[start_idx]["t_s"], df.iloc[i - 1]["t_s"]))
            start_idx = i
            prev = cur
    segs.append((str(prev), df.iloc[start_idx]["t_s"], df.iloc[len(df) - 1]["t_s"]))
    return segs


def add_phase_spans(ax, segs, alpha=0.06):
    colors = {
        "APPROACH": "#4c78a8",
        "FLARE": "#f58518",
        "TOUCHDOWN": "#54a24b",
        "ROLLOUT": "#b279a2",
    }
    for phase, t0, t1 in segs:
        ax.axvspan(t0, t1, color=colors.get(phase, "#999999"), alpha=alpha)


def mark_events(axs, event_times):
    for _, t_evt in event_times.items():
        if t_evt is None:
            continue
        for ax in axs:
            ax.axvline(t_evt, color="#444444", lw=1.0, ls="--", alpha=0.5)


def annotate_mode_changes(axs, frame: pd.DataFrame, mode_col: str = "flare_mode"):
    if mode_col not in frame.columns or frame.empty:
        return
    mode = frame[mode_col].fillna("").astype(str)
    changes = frame[mode.ne(mode.shift().fillna(""))]
    for _, r in changes.iterrows():
        m = str(r[mode_col]).strip()
        if m == "":
            continue
        for ax in axs:
            ax.axvline(r["t_s"], color="#1f77b4", ls=":", lw=1.0, alpha=0.7)
    if SHOW_MODE_LABELS and not changes.empty:
        ylim = axs[0].get_ylim()
        y = ylim[0] + 0.92 * (ylim[1] - ylim[0])
        for _, r in changes.iterrows():
            m = str(r[mode_col]).strip()
            if m == "":
                continue
            axs[0].text(r["t_s"], y, m, rotation=90, fontsize=8, va="top")


def build_flare_frame(df: pd.DataFrame) -> pd.DataFrame:
    flare_mask = df["phase"].isin(["FLARE", "TOUCHDOWN", "ROLLOUT"]) if "phase" in df.columns else pd.Series(False, index=df.index)
    f = df.loc[flare_mask].copy()
    if f.empty:
        return f

    gamma_cmd_col = "flare_fpa_cmd" if ("flare_fpa_cmd" in f.columns and f["flare_fpa_cmd"].notna().any()) else "aa_fpa_cmd_deg"
    f["flare_gamma_cmd"] = col_or_nan(f, gamma_cmd_col)
    f["vs_err_calc"] = col_or_nan(f, "flare_tgt_vs") - col_or_nan(f, "vs_ms")
    f["fpa_err_calc"] = f["flare_gamma_cmd"] - col_or_nan(f, "actual_fpa_deg")
    f["pitch_err_calc"] = col_or_nan(f, "aa_dir_pitch_deg") - col_or_nan(f, "pitch_deg")

    if "flare_vs_err" not in f.columns:
        f["flare_vs_err"] = f["vs_err_calc"]
    if "flare_fpa_err" not in f.columns:
        f["flare_fpa_err"] = f["fpa_err_calc"]
    if "flare_pitch_err" not in f.columns:
        f["flare_pitch_err"] = f["pitch_err_calc"]

    f["cmd_act_pitch_gap"] = col_or_nan(f, "pitch_deg") - col_or_nan(f, "aa_dir_pitch_deg")
    f["aa_cmd_theta_gap"] = col_or_nan(f, "aa_dir_pitch_deg") - col_or_nan(f, "flare_theta_cmd_clamped")

    t = col_or_nan(f, "t_s").to_numpy(dtype=float)
    vs = col_or_nan(f, "vs_ms").to_numpy(dtype=float)
    if len(f) >= 2:
        f["meas_up_a_raw"] = np.gradient(vs, t)
        w = max(int(ACCEL_SMOOTH_WINDOW), 1)
        f["meas_up_a"] = pd.Series(f["meas_up_a_raw"]).rolling(window=w, center=True, min_periods=1).mean().to_numpy()
    else:
        f["meas_up_a_raw"] = np.nan
        f["meas_up_a"] = np.nan

    h_col = "flare_ctrl_h_m" if ("flare_ctrl_h_m" in f.columns and f["flare_ctrl_h_m"].notna().any()) else "gear_h_m"
    f["analysis_h_m"] = np.maximum(col_or_nan(f, h_col), 0.5)

    if "flare_req_up_a" in f.columns and f["flare_req_up_a"].notna().any():
        f["req_up_a"] = col_or_nan(f, "flare_req_up_a")
    else:
        vs_tgt = col_or_nan(f, "flare_tgt_vs")
        f["req_up_a"] = (col_or_nan(f, "vs_ms")**2 - vs_tgt**2) / (2.0 * f["analysis_h_m"])

    f["a_margin"] = f["meas_up_a"] - f["req_up_a"]
    return f



In [ ]:
project_root = detect_project_root(Path.cwd())
logs_dir = Path(LOGS_DIR) if LOGS_DIR is not None else (project_root / "Integrated Flight Computer" / "logs")

log_path = find_latest_log(logs_dir) if USE_LATEST_LOG else (project_root / LOG_FILE_PATH)
df = load_ifc_log(log_path)
segs = phase_segments(df, "phase")
flare_df = build_flare_frame(df)

print(f"Loaded: {log_path}")
print(f"Rows: {len(df):,}")
print(f"Time span: {df['t_s'].min():.2f}s -> {df['t_s'].max():.2f}s")
if safe_col(df, "craft_name"):
    print(f"Craft: {df['craft_name'].iloc[-1]}")
if safe_col(df, "cfg_file"):
    print(f"Cfg: {df['cfg_file'].iloc[-1]}")
print(f"Flare rows: {len(flare_df):,}")

display(df.head(3))


In [ ]:
# Build quick event summary used by plots and findings.
event_times = {
    "first_near_zero_vs": None,
    "first_balloon": None,
    "first_touchdown_phase": None,
}

if flare_df.empty:
    print("No FLARE/TOUCHDOWN/ROLLOUT rows in this log.")
else:
    in_ft = flare_df[flare_df["phase"].isin(["FLARE", "TOUCHDOWN"])]

    print("Phase counts:")
    display(df["phase"].value_counts(dropna=False).rename_axis("phase").to_frame("rows"))

    if "flare_mode" in flare_df.columns:
        print("Flare mode counts:")
        display(flare_df["flare_mode"].replace("", "<blank>").value_counts(dropna=False).rename_axis("flare_mode").to_frame("rows"))

    near_zero = in_ft[np.abs(col_or_nan(in_ft, "vs_ms")) <= VS_NEAR_ZERO_THRESH]
    if not near_zero.empty:
        nz = near_zero.iloc[0]
        event_times["first_near_zero_vs"] = float(nz["t_s"])
        print(f"First |VS| <= {VS_NEAR_ZERO_THRESH:.2f} at t={nz['t_s']:.2f}, gear_h={nz.get('gear_h_m', np.nan):.2f}, agl={nz.get('agl_m', np.nan):.2f}")

    balloon = in_ft[(col_or_nan(in_ft, "vs_ms") >= BALLOON_VS_THRESH) & (col_or_nan(in_ft, "gear_h_m") >= BALLOON_GEAR_H_MIN)]
    if not balloon.empty:
        b0 = balloon.iloc[0]
        bpk = balloon.loc[col_or_nan(balloon, "vs_ms").idxmax()]
        event_times["first_balloon"] = float(b0["t_s"])
        print(f"Balloon: first t={b0['t_s']:.2f}, peak VS={bpk['vs_ms']:.2f} at gear_h={bpk['gear_h_m']:.2f}")

    td = df[df["phase"] == "TOUCHDOWN"] if "phase" in df.columns else pd.DataFrame()
    if not td.empty:
        event_times["first_touchdown_phase"] = float(td.iloc[0]["t_s"])
        print(f"First TOUCHDOWN phase row at t={td.iloc[0]['t_s']:.2f}")



In [ ]:
# ===== Overview plot =====
fig, axs = plt.subplots(5, 1, sharex=True, figsize=(18, 16))

axs[0].plot(df["t_s"], col_or_nan(df, "agl_m"), label="AGL [m]", lw=1.7)
axs[0].plot(df["t_s"], col_or_nan(df, "gear_h_m"), label="Gear Height [m]", lw=1.4)
if safe_col(df, "gs_nom_alt_m"):
    axs[0].plot(df["t_s"], col_or_nan(df, "gs_nom_alt_m"), "--", label="GS Nom Alt [m]", alpha=0.8)
axs[0].axhline(0, color="k", lw=1, alpha=0.45)
axs[0].set_ylabel("Height [m]")
axs[0].legend(loc="best")

axs[1].plot(df["t_s"], col_or_nan(df, "vs_ms"), label="VS [m/s]", lw=1.7)
if safe_col(df, "flare_tgt_vs"):
    axs[1].plot(df["t_s"], col_or_nan(df, "flare_tgt_vs"), "--", label="Flare Target VS", alpha=0.85)
axs[1].axhline(0, color="k", lw=1, alpha=0.45)
axs[1].set_ylabel("VS [m/s]")
axs[1].legend(loc="best")

axs[2].plot(df["t_s"], col_or_nan(df, "ias_ms"), label="IAS [m/s]", lw=1.7)
if safe_col(df, "vapp_ms"):
    axs[2].plot(df["t_s"], col_or_nan(df, "vapp_ms"), "--", label="Vapp/Vtgt [m/s]", alpha=0.8)
axs[2].set_ylabel("Speed [m/s]")
axs[2].legend(loc="best")

axs[3].plot(df["t_s"], col_or_nan(df, "pitch_deg"), label="Pitch [deg]", lw=1.6)
axs[3].plot(df["t_s"], col_or_nan(df, "aoa_deg"), label="AoA [deg]", lw=1.3)
if safe_col(df, "aa_dir_pitch_deg"):
    axs[3].plot(df["t_s"], col_or_nan(df, "aa_dir_pitch_deg"), "--", label="AA Dir Pitch [deg]", alpha=0.9)
axs[3].set_ylabel("Angle [deg]")
axs[3].legend(loc="best")

axs[4].plot(df["t_s"], col_or_nan(df, "thr_cmd"), label="Throttle Cmd", lw=1.7)
if safe_col(df, "thr_cur"):
    axs[4].plot(df["t_s"], col_or_nan(df, "thr_cur"), "--", label="Throttle Cur", alpha=0.8)
if safe_col(df, "flare_thr_floor"):
    axs[4].plot(df["t_s"], col_or_nan(df, "flare_thr_floor"), ":", label="Flare Thr Floor", alpha=0.9)
if safe_col(df, "flare_auth_limited"):
    axs[4].plot(df["t_s"], col_or_nan(df, "flare_auth_limited"), label="Auth Limited", alpha=0.8)
axs[4].set_ylabel("Throttle / Latch")
axs[4].set_xlabel("Time [s]")
axs[4].legend(loc="best")

for ax in axs:
    add_phase_spans(ax, segs)
    ax.grid(True, alpha=0.35)

mark_events(axs, event_times)
fig.suptitle(f"IFC Overview - {log_path.name}", y=0.995)
plt.tight_layout()
plt.show()


In [ ]:
# ===== Flare zoom =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN/ROLLOUT rows to plot.")
else:
    z = flare_df[flare_df["phase"].isin(["FLARE", "TOUCHDOWN"])].copy()
    if z.empty:
        z = flare_df.copy()

    fig, axs = plt.subplots(5, 1, sharex=True, figsize=(18, 17))

    axs[0].plot(z["t_s"], col_or_nan(z, "gear_h_m"), label="Gear Height [m]", lw=1.8)
    axs[0].plot(z["t_s"], col_or_nan(z, "agl_m"), label="AGL [m]", lw=1.2, alpha=0.85)
    if safe_col(z, "flare_ctrl_h_m"):
        axs[0].plot(z["t_s"], col_or_nan(z, "flare_ctrl_h_m"), ":", label="Flare Ctrl H [m]", alpha=0.9)
    axs[0].axhline(0, color="k", lw=1, alpha=0.5)
    axs[0].set_ylabel("Height [m]")
    axs[0].legend(loc="best")

    axs[1].plot(z["t_s"], col_or_nan(z, "vs_ms"), label="VS [m/s]", lw=1.8)
    axs[1].plot(z["t_s"], col_or_nan(z, "flare_tgt_vs"), "--", label="Target VS", alpha=0.9)
    td_rows = df[df["phase"] == "TOUCHDOWN"] if "phase" in df.columns else pd.DataFrame()
    if not td_rows.empty:
        td0 = td_rows.iloc[0]
        td_t = float(td0["t_s"])
        td_vs = float(td0.get("vs_ms", np.nan))
        if np.isfinite(td_vs):
            axs[1].scatter([td_t], [td_vs], s=55, color="magenta", zorder=6, label=f"Touchdown VS: {td_vs:.2f} m/s")
            axs[1].annotate(
                f"TD VS {td_vs:.2f} m/s",
                xy=(td_t, td_vs),
                xytext=(8, 8),
                textcoords="offset points",
                fontsize=9,
                color="magenta",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="magenta", alpha=0.75),
                arrowprops=dict(arrowstyle="->", color="magenta", lw=1),
            )
    axs[1].axhline(0, color="k", lw=1, alpha=0.5)
    axs[1].set_ylabel("VS [m/s]")
    axs[1].legend(loc="best")

    axs[2].plot(z["t_s"], col_or_nan(z, "actual_fpa_deg"), label="Actual FPA [deg]", lw=1.8)
    axs[2].plot(z["t_s"], col_or_nan(z, "flare_gamma_cmd"), "--", label="Gamma Cmd [deg]", alpha=0.9)
    if safe_col(z, "flare_gamma_ref"):
        axs[2].plot(z["t_s"], col_or_nan(z, "flare_gamma_ref"), ":", label="Gamma Ref [deg]", alpha=0.9)
    axs[2].set_ylabel("FPA [deg]")
    axs[2].legend(loc="best")

    axs[3].plot(z["t_s"], col_or_nan(z, "flare_theta_cmd_raw"), label="Theta Cmd Raw [deg]", lw=1.4)
    axs[3].plot(z["t_s"], col_or_nan(z, "flare_theta_cmd_clamped"), label="Theta Cmd Clamped [deg]", lw=1.7)
    axs[3].plot(z["t_s"], col_or_nan(z, "aa_dir_pitch_deg"), "--", label="AA Dir Pitch [deg]", alpha=0.9)
    axs[3].plot(z["t_s"], col_or_nan(z, "pitch_deg"), label="Aircraft Pitch [deg]", lw=1.4)
    axs[3].set_ylabel("Pitch Chain [deg]")
    axs[3].legend(loc="best")

    axs[4].plot(z["t_s"], col_or_nan(z, "aoa_deg"), label="AoA [deg]", lw=1.5)
    axs[4].plot(z["t_s"], col_or_nan(z, "cmd_act_pitch_gap"), label="Pitch - Dir Pitch [deg]", lw=1.4)
    axs[4].plot(z["t_s"], col_or_nan(z, "aa_cmd_theta_gap"), label="Dir Pitch - ThetaCmd [deg]", lw=1.2)
    axs[4].axhline(0, color="k", lw=1, alpha=0.5)
    axs[4].set_ylabel("Gap / AoA")
    axs[4].set_xlabel("Time [s]")
    axs[4].legend(loc="best")

    for ax in axs:
        ax.grid(True, alpha=0.35)

    annotate_mode_changes(axs, z, "flare_mode")
    mark_events(axs, event_times)

    fig.suptitle(f"Flare and Touchdown Zoom - {log_path.name}", y=0.995)
    plt.tight_layout()
    plt.show()


In [ ]:
# ===== Control authority diagnostics =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN/ROLLOUT rows to analyze.")
else:
    has_elev_cols = all(c in flare_df.columns for c in ["flare_pitch_in_cmd", "elev_defl_avg_deg", "elev_defl_max_deg", "elev_defl_n"])
    z = flare_df[flare_df["phase"].isin(["FLARE", "TOUCHDOWN"])].copy()
    if z.empty:
        z = flare_df.copy()

    nrows = 5 if has_elev_cols else 4
    fig, axs = plt.subplots(nrows, 1, sharex=True, figsize=(18, 4.0 * nrows))

    axs[0].plot(z["t_s"], col_or_nan(z, "flare_pitch_err"), label="Pitch Err [deg]", lw=1.8)
    axs[0].plot(z["t_s"], col_or_nan(z, "flare_fpa_err"), label="FPA Err [deg]", lw=1.4)
    axs[0].plot(z["t_s"], col_or_nan(z, "flare_vs_err"), label="VS Err [m/s]", lw=1.4)
    axs[0].axhline(0, color="k", lw=1, alpha=0.5)
    axs[0].set_ylabel("Tracking Error")
    axs[0].legend(loc="best")
    axs[0].grid(True, alpha=0.35)

    axs[1].plot(z["t_s"], col_or_nan(z, "cmd_act_pitch_gap"), label="Pitch - Dir Pitch [deg]", lw=1.8)
    axs[1].plot(z["t_s"], col_or_nan(z, "aa_cmd_theta_gap"), label="Dir Pitch - ThetaCmd [deg]", lw=1.5)
    axs[1].axhline(0, color="k", lw=1, alpha=0.5)
    axs[1].set_ylabel("Command Gaps")
    axs[1].legend(loc="best")
    axs[1].grid(True, alpha=0.35)

    axs[2].plot(z["t_s"], col_or_nan(z, "flare_auth_limited"), label="Auth Limited", lw=1.6)
    axs[2].plot(z["t_s"], col_or_nan(z, "flare_theta_clamp_active"), label="Theta Clamp Active", lw=1.4)
    axs[2].plot(z["t_s"], col_or_nan(z, "flare_aoa_clamp_active"), label="AoA Clamp Likely", lw=1.4)
    axs[2].plot(z["t_s"], col_or_nan(z, "flare_thr_floor"), label="Thr Floor", lw=1.2)
    axs[2].plot(z["t_s"], col_or_nan(z, "thr_cmd"), label="Thr Cmd", lw=1.2)
    axs[2].set_ylabel("Latch / Limits")
    axs[2].legend(loc="best")
    axs[2].grid(True, alpha=0.35)

    axs[3].plot(z["t_s"], col_or_nan(z, "aa_dir"), label="AA Director On", lw=1.4)
    axs[3].plot(z["t_s"], col_or_nan(z, "aa_fbw"), label="AA FBW On", lw=1.4)
    axs[3].set_ylabel("AA Modes")
    axs[3].legend(loc="best")
    axs[3].grid(True, alpha=0.35)

    if has_elev_cols:
        axs[4].plot(z["t_s"], col_or_nan(z, "flare_pitch_in_cmd"), label="Pitch Input Cmd [-1..1]", lw=1.6)
        axs[4].plot(z["t_s"], col_or_nan(z, "elev_defl_avg_deg"), label="Elev Defl Avg |deg|", lw=1.4)
        axs[4].plot(z["t_s"], col_or_nan(z, "elev_defl_max_deg"), label="Elev Defl Max |deg|", lw=1.4)
        axs[4].set_ylabel("Cmd / Deflection")
        axs[4].legend(loc="best")
        axs[4].grid(True, alpha=0.35)

    axs[-1].set_xlabel("Time [s]")
    annotate_mode_changes(axs, z, "flare_mode")
    mark_events(axs, event_times)

    fig.suptitle(f"Control Authority Diagnostics - {log_path.name}", y=0.995)
    plt.tight_layout()
    plt.show()

    if "flare_auth_reason" in z.columns:
        reason_counts = z["flare_auth_reason"].replace("", "<blank>").value_counts(dropna=False)
        print("flare_auth_reason counts:")
        display(reason_counts.rename_axis("reason").to_frame("rows"))


## TECS and Command Pipeline Diagnostics

This section overlays TECS energy channels with flare command-chain signals:
- gamma/fpa reference and command
- theta raw/clamped and AA director pitch
- aircraft pitch response and throttle floor/latch behavior


In [ ]:
# ===== TECS / energy plots =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN/ROLLOUT rows to analyze.")
else:
    z = flare_df[flare_df["phase"].isin(["FLARE", "TOUCHDOWN"])].copy()
    if z.empty:
        z = flare_df.copy()

    g = 9.81
    h_col = "flare_ctrl_h_m" if ("flare_ctrl_h_m" in z.columns and z["flare_ctrl_h_m"].notna().any()) else "gear_h_m"

    z["et_now_est"] = 0.5 * col_or_nan(z, "ias_ms")**2 + g * np.maximum(col_or_nan(z, h_col), 0.0)
    z["eb_now_est"] = g * np.maximum(col_or_nan(z, h_col), 0.0) - 0.5 * col_or_nan(z, "ias_ms")**2
    z["et_ref_est"] = 0.5 * col_or_nan(z, "flare_tecs_v_ref")**2 + g * np.maximum(col_or_nan(z, "flare_tecs_h_ref"), 0.0)
    z["eb_ref_est"] = g * np.maximum(col_or_nan(z, "flare_tecs_h_ref"), 0.0) - 0.5 * col_or_nan(z, "flare_tecs_v_ref")**2

    fig, axs = plt.subplots(4, 1, sharex=True, figsize=(18, 15))

    axs[0].plot(z["t_s"], col_or_nan(z, "flare_tecs_et_err"), label="Et Error", lw=1.8)
    axs[0].plot(z["t_s"], col_or_nan(z, "flare_tecs_eb_err"), label="Eb Error", lw=1.8)
    axs[0].axhline(0, color="k", lw=1, alpha=0.5)
    axs[0].set_ylabel("Error [m^2/s^2]")
    axs[0].set_title("TECS Error Channels")
    axs[0].legend(loc="best")
    axs[0].grid(True, alpha=0.35)

    axs[1].plot(z["t_s"], z["et_ref_est"], label="Et Ref (est)", lw=1.5)
    axs[1].plot(z["t_s"], z["et_now_est"], label="Et Now (est)", lw=1.5)
    axs[1].plot(z["t_s"], z["eb_ref_est"], label="Eb Ref (est)", lw=1.2, alpha=0.9)
    axs[1].plot(z["t_s"], z["eb_now_est"], label="Eb Now (est)", lw=1.2, alpha=0.9)
    axs[1].set_ylabel("Energy [m^2/s^2]")
    axs[1].set_title(f"Estimated Energy State ({h_col})")
    axs[1].legend(loc="best")
    axs[1].grid(True, alpha=0.35)

    axs[2].plot(z["t_s"], col_or_nan(z, "flare_gamma_ref"), label="Gamma Ref [deg]", lw=1.3)
    axs[2].plot(z["t_s"], col_or_nan(z, "flare_gamma_cmd"), label="Gamma Cmd [deg]", lw=1.8)
    axs[2].plot(z["t_s"], col_or_nan(z, "actual_fpa_deg"), label="Actual FPA [deg]", lw=1.5)
    axs[2].plot(z["t_s"], col_or_nan(z, "flare_theta_cmd_clamped"), label="Theta Cmd Clamped [deg]", lw=1.4)
    axs[2].plot(z["t_s"], col_or_nan(z, "aa_dir_pitch_deg"), "--", label="AA Dir Pitch [deg]", alpha=0.9)
    axs[2].plot(z["t_s"], col_or_nan(z, "pitch_deg"), label="Pitch [deg]", lw=1.2)
    axs[2].axhline(0, color="k", lw=1, alpha=0.5)
    axs[2].set_ylabel("Gamma / Pitch [deg]")
    axs[2].set_title("Command Chain")
    axs[2].legend(loc="best")
    axs[2].grid(True, alpha=0.35)

    axs[3].plot(z["t_s"], col_or_nan(z, "thr_cmd"), label="Throttle Cmd", lw=1.8)
    axs[3].plot(z["t_s"], col_or_nan(z, "flare_thr_floor"), label="Thr Floor", lw=1.4)
    axs[3].plot(z["t_s"], col_or_nan(z, "flare_auth_limited"), label="Auth Limited", lw=1.2)
    axs[3].plot(z["t_s"], col_or_nan(z, "flare_theta_clamp_active"), label="Theta Clamp", lw=1.2)
    axs[3].plot(z["t_s"], col_or_nan(z, "flare_aoa_clamp_active"), label="AoA Clamp Likely", lw=1.2)
    axs[3].set_ylabel("Throttle / Latch")
    axs[3].set_xlabel("Time [s]")
    axs[3].set_title("Throttle and Limit Channels")
    axs[3].legend(loc="best")
    axs[3].grid(True, alpha=0.35)

    annotate_mode_changes(axs, z, "flare_mode")
    mark_events(axs, event_times)

    fig.suptitle(f"TECS and Command Diagnostics - {log_path.name}", y=0.995)
    plt.tight_layout()
    plt.show()

    summary = pd.Series({
        "Et err mean": col_or_nan(z, "flare_tecs_et_err").mean(),
        "Et err abs max": col_or_nan(z, "flare_tecs_et_err").abs().max(),
        "Eb err mean": col_or_nan(z, "flare_tecs_eb_err").mean(),
        "Eb err abs max": col_or_nan(z, "flare_tecs_eb_err").abs().max(),
        "Cmd-Act pitch gap mean": col_or_nan(z, "cmd_act_pitch_gap").mean(),
        "Cmd-Act pitch gap abs max": col_or_nan(z, "cmd_act_pitch_gap").abs().max(),
    })
    display(summary.round(3))


## Vertical Acceleration Authority Margin

Compares required upward acceleration (to reach target VS over remaining height)
against measured d(VS)/dt.


In [ ]:
# ===== Vertical acceleration authority margin =====
if flare_df.empty:
    print("No FLARE/TOUCHDOWN/ROLLOUT rows to analyze.")
else:
    z = flare_df[flare_df["phase"].isin(["FLARE", "TOUCHDOWN"])].copy()
    if z.empty:
        z = flare_df.copy()

    needed = ["t_s", "vs_ms", "flare_tgt_vs", "req_up_a", "meas_up_a", "a_margin", "analysis_h_m"]
    missing = [c for c in needed if c not in z.columns]
    if missing:
        print("Missing columns for accel-margin diagnostics:", missing)
    else:
        focus = (col_or_nan(z, "vs_ms") < -0.2) & (col_or_nan(z, "analysis_h_m") > BALLOON_GEAR_H_MIN)
        if focus.any():
            neg_frac = (col_or_nan(z.loc[focus], "a_margin") < 0).mean()
            print(f"Negative accel margin while descending above {BALLOON_GEAR_H_MIN:.1f} m: {100.0 * neg_frac:.1f}%")
        else:
            print("No focus rows (descending above configured height floor).")

        fig, axs = plt.subplots(3, 1, sharex=True, figsize=(18, 12))

        axs[0].plot(z["t_s"], col_or_nan(z, "req_up_a"), label="Required Up Accel [m/s^2]", lw=1.8)
        axs[0].plot(z["t_s"], col_or_nan(z, "meas_up_a"), label=f"Measured Up Accel [m/s^2] (smooth={ACCEL_SMOOTH_WINDOW})", lw=1.8)
        axs[0].axhline(0, color="k", lw=1, alpha=0.5)
        axs[0].set_ylabel("Accel [m/s^2]")
        axs[0].set_title("Required vs Measured Vertical Acceleration")
        axs[0].legend(loc="best")
        axs[0].grid(True, alpha=0.35)

        margin = col_or_nan(z, "a_margin")
        axs[1].plot(z["t_s"], margin, label="Accel Margin (meas - req)", lw=1.8)
        axs[1].fill_between(z["t_s"], margin, 0, where=(margin < 0), alpha=0.2, color="red")
        axs[1].axhline(0, color="k", lw=1, alpha=0.5)
        axs[1].set_ylabel("Margin [m/s^2]")
        axs[1].set_title("Vertical Acceleration Margin")
        axs[1].legend(loc="best")
        axs[1].grid(True, alpha=0.35)

        axs[2].plot(z["t_s"], col_or_nan(z, "vs_ms"), label="VS [m/s]", lw=1.8)
        axs[2].plot(z["t_s"], col_or_nan(z, "flare_tgt_vs"), "--", label="Target VS [m/s]", alpha=0.9)
        axs[2].plot(z["t_s"], col_or_nan(z, "cmd_act_pitch_gap"), label="Pitch - Dir Pitch [deg]", alpha=0.85)
        axs[2].axhline(0, color="k", lw=1, alpha=0.5)
        axs[2].set_ylabel("Context")
        axs[2].set_xlabel("Time [s]")
        axs[2].set_title("Context: VS Tracking and Pitch Gap")
        axs[2].legend(loc="best")
        axs[2].grid(True, alpha=0.35)

        annotate_mode_changes(axs, z, "flare_mode")
        mark_events(axs, event_times)

        fig.suptitle(f"Vertical Acceleration Margin - {log_path.name}", y=0.995)
        plt.tight_layout()
        plt.show()


In [ ]:
# ===== Automatic issue summary =====
issues = []
notes = []

if flare_df.empty:
    print("No flare segment found; cannot run flare diagnostics.")
else:
    z = flare_df[flare_df["phase"].isin(["FLARE", "TOUCHDOWN"])].copy()
    if z.empty:
        z = flare_df.copy()

    near_zero = z[np.abs(col_or_nan(z, "vs_ms")) <= VS_NEAR_ZERO_THRESH]
    if not near_zero.empty:
        nz = near_zero.iloc[0]
        if nz.get("gear_h_m", np.nan) >= EARLY_ARREST_GEAR_H:
            issues.append(f"Early sink arrest: first |VS| <= {VS_NEAR_ZERO_THRESH:.2f} at gear_h={nz['gear_h_m']:.2f} m (t={nz['t_s']:.2f}).")
        else:
            notes.append(f"Near-zero VS occurs low: gear_h={nz.get('gear_h_m', np.nan):.2f} m (t={nz['t_s']:.2f}).")

    balloon_rows = z[(col_or_nan(z, "vs_ms") >= BALLOON_VS_THRESH) & (col_or_nan(z, "gear_h_m") >= BALLOON_GEAR_H_MIN)]
    if not balloon_rows.empty:
        peak = balloon_rows.loc[col_or_nan(balloon_rows, "vs_ms").idxmax()]
        issues.append(f"Balloon detected: VS peak {peak['vs_ms']:.2f} m/s at gear_h={peak['gear_h_m']:.2f} m (t={peak['t_s']:.2f}).")

    bad_pitch = z[col_or_nan(z, "flare_pitch_err") <= -PERSISTENT_TRACK_ERR_DEG]
    frac_bad_pitch = len(bad_pitch) / max(len(z), 1)
    if frac_bad_pitch >= 0.20:
        issues.append(f"Persistent pitch tracking miss: {frac_bad_pitch*100:.1f}% rows have pitch_err <= -{PERSISTENT_TRACK_ERR_DEG:.1f} deg.")
    else:
        notes.append(f"Pitch tracking miss fraction: {frac_bad_pitch*100:.1f}%.")

    pitch_gap = col_or_nan(z, "cmd_act_pitch_gap").abs()
    gap_frac = (pitch_gap >= CMD_ACT_PITCH_GAP_WARN).mean()
    if gap_frac >= 0.20:
        issues.append(f"Large commanded-vs-actual pitch gap: {gap_frac*100:.1f}% rows exceed {CMD_ACT_PITCH_GAP_WARN:.1f} deg.")

    if "aa_cmd_theta_gap" in z.columns and z["aa_cmd_theta_gap"].notna().any():
        theta_gap_frac = (z["aa_cmd_theta_gap"].abs() >= AA_CMD_THETA_GAP_WARN).mean()
        if theta_gap_frac >= 0.10:
            issues.append(f"AA command-chain mismatch: {theta_gap_frac*100:.1f}% rows have |aa_dir_pitch-theta_cmd| >= {AA_CMD_THETA_GAP_WARN:.2f} deg.")

    if "flare_theta_clamp_active" in z.columns:
        clamp_frac = (col_or_nan(z, "flare_theta_clamp_active") > 0).mean()
        notes.append(f"Theta clamp active for {clamp_frac*100:.1f}% of rows.")

    if "flare_auth_limited" in z.columns:
        auth_frac = (col_or_nan(z, "flare_auth_limited") > 0).mean()
        notes.append(f"Authority-limited active for {auth_frac*100:.1f}% of rows.")

    if "flare_auth_reason" in z.columns:
        reason_counts = z["flare_auth_reason"].replace("", "<blank>").value_counts(dropna=False)
        notes.append("Auth reasons: " + ", ".join([f"{k}={v}" for k, v in reason_counts.items()][:6]))

    if "a_margin" in z.columns and z["a_margin"].notna().any():
        focus = (col_or_nan(z, "vs_ms") < -0.2) & (col_or_nan(z, "analysis_h_m") > BALLOON_GEAR_H_MIN)
        if focus.any():
            neg_frac = (col_or_nan(z.loc[focus], "a_margin") < 0).mean()
            if neg_frac >= NEG_ACCEL_MARGIN_WARN_FRAC:
                issues.append(f"Insufficient vertical acceleration margin: negative for {neg_frac*100:.1f}% of focus rows.")
            else:
                notes.append(f"Negative accel-margin in focus rows: {neg_frac*100:.1f}%.")

print("=== Auto Findings ===")
if issues:
    for i, item in enumerate(issues, start=1):
        print(f"{i}. {item}")
else:
    print("No major flare anomalies flagged by current thresholds.")

print("\n=== Notes ===")
for item in notes:
    print(f"- {item}")


In [ ]:
# Optional schema explorer for troubleshooting log format changes
schema = pd.DataFrame({"column": sorted(df.columns)})
display(schema)

for p in ["flare_", "aa_", "asc_", "ifc_"]:
    cols = [c for c in df.columns if c.startswith(p)]
    print(f"{p} columns: {len(cols)}")
    if cols:
        print("  " + ", ".join(cols))
